In [1]:
# 01_credit_donor_controls.ipynb
# Donor-domain demonstration (German Credit): proxy detection (T-1) and a
# decision-level explanation / recourse example (T-2) for an automated credit
# decision. All code is contained in this single cell.

import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# ---- resolve project folders without printing any path (anonymity) ----
def find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "results").is_dir() and (c / "notebooks").is_dir():
            return c
    return p.parent if p.name == "notebooks" else p
ROOT = find_root()
DATA, FIG, TAB = ROOT / "data", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

# ---- grayscale publication style; figures saved as png + pdf at 600 dpi ----
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 600, "font.size": 11,
                     "axes.edgecolor": "0.2", "grid.color": "0.85"})
G_DARK, G_LIGHT = "0.25", "0.65"
def savefig(fig, stem):
    for e in ("png", "pdf"):
        fig.savefig(FIG / (stem + "." + e), dpi=600, bbox_inches="tight")
    plt.close(fig)
def savetable(df, stem, index=False):
    df.to_csv(TAB / (stem + ".csv"), index=index)
    try:
        df.to_latex(TAB / (stem + ".tex"), index=index, float_format="%.3f")
    except Exception:
        pass

# ---- load (download public mirror if absent) ----
fp = DATA / "credit_german.csv"
if not fp.exists():
    url = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
    pd.read_csv(url).to_csv(fp, index=False)
df = pd.read_csv(fp)
df.columns = [c.strip().strip('"') for c in df.columns]

# ---- protected attribute + target ----
df["sex"] = np.where(df["personal_status_sex"].astype(str).str.contains("female"),
                     "female", "male")
y = (df["credit_risk"].astype(int) == 1).astype(int)   # 1 = good credit (approve)
X = df.drop(columns=["credit_risk", "personal_status_sex"])
num_cols = X.select_dtypes("number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols and c != "sex"]
feat = [c for c in X.columns if c != "sex"]

def make_pre(cols):
    return ColumnTransformer([
        ("num", StandardScaler(), [c for c in num_cols if c in cols]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in cat_cols if c in cols])])

# ---- (A) credit-decision model + selection-rate disparity by sex ----
clf = Pipeline([("pre", make_pre(feat)),
                ("gb", GradientBoostingClassifier(random_state=SEED))])
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
proba = cross_val_predict(clf, X[feat], y, cv=cv, method="predict_proba")[:, 1]
pred = (proba >= 0.5).astype(int)
grp = pd.DataFrame({"sex": df["sex"], "approved": pred, "good": y})
rate = grp.groupby("sex")["approved"].mean()
di_ratio = rate.min() / rate.max()
gmetrics = grp.groupby("sex").agg(n=("approved", "size"),
            selection_rate=("approved", "mean"),
            base_rate=("good", "mean")).reset_index()
gmetrics["disparate_impact_ratio"] = di_ratio
savetable(gmetrics, "credit_group_metrics")

# ---- (B) proxy detection (T-1): recover sex from non-sex features ----
sx = (df["sex"] == "female").astype(int)
proxy = Pipeline([("pre", make_pre(feat)),
                  ("lr", LogisticRegression(max_iter=2000))])
proxy_p = cross_val_predict(proxy, X[feat], sx, cv=cv, method="predict_proba")[:, 1]
proxy_auc = roc_auc_score(sx, proxy_p)
savetable(pd.DataFrame({"quantity": ["proxy_recovery_auc", "chance"],
                        "value": [proxy_auc, 0.5]}), "credit_proxy_auc")

# ---- (C) recourse (T-2): minimal change to flip one declined applicant ----
clf.fit(X[feat], y)
declined = np.where(pred == 0)[0]
rec = {"recourse": "none_found"}
if len(declined):
    i = int(declined[0])
    base = X[feat].iloc[[i]].copy()
    p0 = clf.predict_proba(base)[0, 1]
    if "duration" in feat:
        for delta in range(0, 49, 3):
            trial = base.copy()
            trial["duration"] = max(0, base["duration"].values[0] - delta)
            if clf.predict_proba(trial)[0, 1] >= 0.5:
                rec = {"feature": "duration", "reduce_by_months": delta,
                       "p_before": round(float(p0), 3)}
                break
savetable(pd.DataFrame([rec]), "credit_recourse_example")

# ---- figures ----
fig, ax = plt.subplots(figsize=(4.2, 3.2))
sns.barplot(data=gmetrics, x="sex", y="selection_rate", ax=ax,
            palette=[G_DARK, G_LIGHT], edgecolor="black")
ax.set_ylabel("Approval (selection) rate"); ax.set_xlabel(""); ax.set_ylim(0, 1)
savefig(fig, "fig_credit_selection_rate")

fpr, tpr, _ = roc_curve(sx, proxy_p)
fig, ax = plt.subplots(figsize=(4.2, 3.4))
ax.plot(fpr, tpr, color=G_DARK, lw=1.8)
ax.plot([0, 1], [0, 1], color=G_LIGHT, ls="--", lw=1.2)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.text(0.52, 0.10, "AUC = %.3f" % proxy_auc)
savefig(fig, "fig_credit_proxy_roc")

print("credit leg done | approval rate:", {k: round(v, 3) for k, v in rate.items()},
      "| DI ratio: %.3f" % di_ratio, "| proxy AUC: %.3f" % proxy_auc)


credit leg done | approval rate: {'female': 0.755, 'male': 0.786} | DI ratio: 0.961 | proxy AUC: 0.698
